# Prepare image request GeoJSON

#### A subset that contains SME request parameters for image acquisaitions at sites from a vendor using the CSDA Evaluation Sites GeoJSON

Paul Montesano, PhD  
June 2026

In [1]:
import pandas as pd
import geopandas as gpd
from datetime import datetime

### Read the CSDA/Eval Sites GeoJSON stored on GitHub

+ This GeoJSON is built directly off the CSDA Evaluation Sites Database.  
+ The notebook to process this GeoJSON is here: https://github.com/pahbs/csda_summaries/blob/master/notebooks/csda_eval_sites_process.ipynb

In [2]:
TYPE = 'eval' #'csda'

In [3]:
RAW_BASE = 'https://raw.githubusercontent.com/pahbs/csda_summaries/master'
sites_url = f'{RAW_BASE}/sites/{TYPE}_sites_aoi.geojson'
sites = gpd.read_file(sites_url)

/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3.8/site-packages/pyproj/../../.././libtiff.so.6: version `LIBTIFF_4.6.1' not found (required by /app/jupyter/ilab/jupyter-lab/prod/lib/gdalplugins/../libgdal.so.36)
/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3.8/site-packages/pyproj/../../.././libtiff.so.6: version `LIBTIFF_4.6.1' not found (required by /app/jupyter/ilab/jupyter-lab/prod/lib/gdalplugins/../libgdal.so.36)
/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3.8/site-packages/pyproj/../../.././libtiff.so.6: version `LIBTIFF_4.6.1' not found (required by /app/jupyter/ilab/jupyter-lab/prod/lib/gdalplugins/../libgdal.so.36)
/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3.8/site-packages/pyproj/../../.././libtiff.so.6: version `LIBTIFF_4.6.1' not found (required by /app/jupyter/ilab/jupyter-lab/prod/lib/gdalplugins/../libgdal.so.36)
/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3

### Indicate a name for the vendor

In [4]:
VENDOR_NAME = 'Tanager' # Change this
VENDOR_NAME = 'satellogic_v2' # Change this
VENDOR_NAME = 'HydroSat'
VENDOR_NAME = 'Airbus'


In [5]:
# Get today's date
DATE = datetime.now().strftime('%Y%m%d')
DATE

'20260922'

In [6]:
#sites.info()

### Check some useful site attributes

In [7]:
print(list(sites['Evaluation Category'].unique()))

['Geometric', 'Radiometric', 'Radiometric & Geometric', 'InSAR', 'Geometric (vert/horiz)', 'All']


### Each site's 'Site Name' is the key identifier for indicating the location of a CSDA request of data from a vendor

In [8]:
print(list(sites['Site Name'].unique()))

['Baoshan', 'Cairns', 'Copiapo', 'Luanda', 'Manta', 'Mexico City', 'Mogadishu', 'Perth', 'Albuquerque', 'Amazon', 'Baotou', 'Atacama Desert', 'Belo Horizonte', 'Boston', 'Kansas City', 'San Diego', 'Los Angeles', 'Novosibirsk', 'Konya-airport', 'Konya-city', 'Mansoura', 'Cape Town', 'Casablanca', 'Caspian Sea', 'Catania', 'Crater Lake', 'Cuprite', 'Antarctica GPS', 'Doldrums', 'Atlantic Doldrums', 'Dublin', 'Gobabeb', 'Gobabeb (Thermal)', 'Petermann Glacier', 'NISAR CR Array', 'Hohhot', 'Etang de Berre', 'La Crau TIR', 'King Fahd Causeway', 'Lake Pontchartrain Causeway', 'Old Bahia Bridge', 'Navarre Causeway', 'Suramadu Bridge', 'London', 'Melbourne', 'DLR CR Array', 'Arabian Peninsula', 'Phoenix', 'La Crau', 'PICS Algeria-3', 'PICS Libya-1', 'PICS Libya-4', 'Piedmont', 'Railroad Valley', 'Lageren', 'Rio Gallegos', 'RCRA', 'Salon-de-Provence', 'Sapporo', 'Shadnagar', 'Singapore', 'Sioux Falls', 'FMI CR Array', 'Valencia', 'Golmud', 'OPERA CR Array', 'WLEF', 'Lake Constance', 'Lake Kasu

#### Function to update site attributes

In [9]:
def update_sites_attributes(sites_gdf, site_configs):
    """
    Update sites GeoDataFrame with attributes based on configuration.
    
    Parameters:
    -----------
    sites_gdf : GeoDataFrame
        Sites geodataframe to update
    site_configs : list of dict
        List of configurations, each with 'sites' and 'order_parameters' keys
        
    Returns:
    --------
    GeoDataFrame : Updated sites (copy)
    list : All site names from configs
    """
    sites_updated = sites_gdf.copy()
    all_sites = []
    
    for config in site_configs:
        site_list = config['sites']
        attributes = config['order_parameters']
        
        # Update attributes for these sites
        mask = sites_updated['Site Name'].isin(site_list)
        for key, value in attributes.items():
            sites_updated.loc[mask, key] = value
        
        all_sites.extend(site_list)
    
    return sites_updated, all_sites

def print_site_update_report(sites_original, sites_updated, site_configs):
    """
    Print a report showing what attributes were updated for which sites.
    
    Parameters:
    -----------
    sites_original : GeoDataFrame
        Original sites before updates
    sites_updated : GeoDataFrame
        Sites after updates
    site_configs : list of dict
        Configuration used for updates
    """
    print("=" * 70)
    print("SITE ATTRIBUTE UPDATE REPORT")
    print("=" * 70)
    
    # Get all unique attributes being updated
    all_attributes = set()
    for config in site_configs:
        all_attributes.update(config['order_parameters'].keys())
    
    total_sites = 0
    
    for i, config in enumerate(site_configs, 1):
        sites_list = config['sites']
        attributes = config['order_parameters']
        
        print(f"\nGroup {i}: {len(sites_list)} site(s)")
        print("-" * 70)
        
        for site in sites_list:
            total_sites += 1
            print(f"\n  Site: {site}")
            
            # Get before/after values
            orig_row = sites_original[sites_original['Site Name'] == site]
            updated_row = sites_updated[sites_updated['Site Name'] == site]
            
            if len(orig_row) == 0:
                print(f"    ⚠️  WARNING: Site not found in original dataframe")
                continue
            
            for attr, new_value in attributes.items():
                old_value = orig_row[attr].iloc[0] if attr in orig_row.columns else 'N/A'
                actual_value = updated_row[attr].iloc[0] if len(updated_row) > 0 else 'ERROR'
                
                # Check if update was successful
                if str(actual_value) == str(new_value):
                    status = "✓"
                else:
                    status = "✗"
                
                print(f"    {status} {attr:20s}: {old_value} → {new_value}")
    
    print("\n" + "=" * 70)
    print(f"Total sites updated: {total_sites}")
    print("=" * 70)


### Update config of request parameters for sites chosen for this vendor request

Here is where we config & specify our 'timeseries' sites and any other types of sites we need to config & specify for this request

In [10]:
import pandas as pd
import re

def get_site_configs_from_sheets(sheet_url, gid=None):
    # Convert standard edit URL to direct CSV export URL
    csv_url = re.sub(r'/edit.*', '/export?format=csv', sheet_url)
    
    # ── Append tab gid if specified ───────────────────────────────────────────
    if gid is not None:
        csv_url += f'&gid={gid}'
    
    df = pd.read_csv(csv_url)
    
    # 1. Define flexible keyword rules for each output parameter
    # The dictionary maps internal output keys to a list of potential matching keywords
    column_rules = {
        "site_name": ["sitename", "site"],
        "location_name": ["locationname", "location"],
        "country": ["country", "nation"],
        "longitude": ["longitude", "long", "lon"],
        "latitude": ["latitude", "lat"],
        "remote_sensing_domain": ["remotesensing", "domain", "sensortype"],
        "evaluation_category": ["evaluation", "category", "eval"],
        "assessment_types": ["assessment", "type"],
        "aoi_shape": ["aoishape", "shape", "geometry"],
        "max_aoi_cloud_pct": ["maxaoicloud", "aoicloud"],
        "max_scene_cloud_pct": ["maxscenecloud", "scenecloud"],
        "aoi_size_km": ["aoisize", "size"],
        "max_view_angle": ["viewangle", "angle", "pointingangle"],
        "min_num_acqs": ["minnumacqs", "minacq", "minimum"],
        "ideal_num_acqs": ["idealnumacqs", "idealacq", "ideal"]
    }
    
    # 2. Dynamic column mapping generation
    # Normalize sheet headers: "max AOI cloud %" becomes "maxaoicloud"
    normalized_headers = {
        re.sub(r'[^a-z0-9]', '', str(col).lower()): col for col in df.columns
    }
    
    resolved_mapping = {}
    for param_key, keywords in column_rules.items():
        for norm_header, raw_header in normalized_headers.items():
            # If any of our fallback keywords are found inside the normalized header, map it
            if any(kw in norm_header for kw in keywords):
                resolved_mapping[param_key] = raw_header
                break  # Stop searching for this parameter once matched

    # Fallback to make sure critical fields exist or won't crash the script
    site_col = resolved_mapping.get("site_name")
    if not site_col:
        raise ValueError(f"Could not find a column representing 'Site Name'. Headers available: {list(df.columns)}")

    site_configs = []
    
    # 3. Process Rows using the dynamic map
    for _, row in df.iterrows():
        if pd.isna(row[site_col]):
            continue
            
        site_name = str(row[site_col]).strip()
        
        def clean_numeric(val, is_float=False):
            if pd.isna(val): return 0
            cleaned = re.sub(r'[^\d\.\-]', '', str(val))
            if not cleaned: return 0
            return float(cleaned) if is_float else int(float(cleaned))

        def safe_get(key, default_val="", is_numeric=False, is_float=False):
            actual_col = resolved_mapping.get(key)
            if actual_col is None or actual_col not in row:
                return 0 if is_numeric else default_val
            return clean_numeric(row[actual_col], is_float) if is_numeric else str(row[actual_col]).strip()

        # Safely extract values using the flexible column dictionary
        order_params = {
            "location_name": safe_get("location_name"),
            "country": safe_get("country"),
            "longitude": safe_get("longitude", is_numeric=True, is_float=True),
            "latitude": safe_get("latitude", is_numeric=True, is_float=True),
            "remote_sensing_domain": safe_get("remote_sensing_domain"),
            "evaluation_category": safe_get("evaluation_category"),
            "assessment_types": safe_get("assessment_types"),
            "aoi_shape": safe_get("aoi_shape"),
            "max_aoi_cloud_pct": safe_get("max_aoi_cloud_pct", is_numeric=True),
            "max_scene_cloud_pct": safe_get("max_scene_cloud_pct", is_numeric=True),
            "aoi_size_km": safe_get("aoi_size_km", is_numeric=True, is_float=True),
            "max_view_angle": safe_get("max_view_angle", is_numeric=True),
            "min_num_acqs": safe_get("min_num_acqs", is_numeric=True),
            "ideal_num_acqs": safe_get("ideal_num_acqs", is_numeric=True)
        }
        
        site_configs.append({
            "sites": [site_name],
            "order_parameters": order_params
        })
        
    return site_configs

In [26]:
# --- Execution ---
SHEET_URL = "https://docs.google.com/spreadsheets/d/1lkR7cnsqq1EDISoca8tpwcdtfPvWNerbJ5EzCgq-9UI/edit?usp=sharing"
SHEET_URL = 'https://docs.google.com/spreadsheets/d/16VopHRqlt5qbl4HgZdkVLUIHLVi9lZWPLJMY-QWsJ8g/edit?usp=sharing'
SHEET_URL = 'https://docs.google.com/spreadsheets/d/16VopHRqlt5qbl4HgZdkVLUIHLVi9lZWPLJMY-QWsJ8g/edit?usp=sharing'
GID=0
# Airbus
SHEET_URL = 'https://docs.google.com/spreadsheets/d/16VopHRqlt5qbl4HgZdkVLUIHLVi9lZWPLJMY-QWsJ8g/edit?usp=sharing'
GID=803290261

SITE_CONFIGS = get_site_configs_from_sheets(SHEET_URL, gid=GID)


In [28]:

sites_updated, SITES_FOR_REQUEST = update_sites_attributes(sites[['Site Name','geometry']], SITE_CONFIGS)

print_site_update_report(sites, sites_updated, SITE_CONFIGS)

SITE ATTRIBUTE UPDATE REPORT

Group 1: 1 site(s)
----------------------------------------------------------------------

  Site: Baotou
    ✓ location_name       : N/A → China cal/val
    ✓ country             : N/A → China
    ✓ longitude           : N/A → 109.629437
    ✓ latitude            : N/A → 40.851787
    ✓ remote_sensing_domain: N/A → Optical Multi/Hyper
    ✓ evaluation_category : N/A → Radiometric & Geometric
    ✓ assessment_types    : N/A → 
    ✓ aoi_shape           : box → box
    ✗ max_aoi_cloud_pct   : N/A → 0
    ✗ max_scene_cloud_pct : N/A → 0
    ✓ aoi_size_km         : 0.04 → 0.04
    ✗ max_view_angle      : 15.0 → 15
    ✗ min_num_acqs        : nan → 1
    ✗ ideal_num_acqs      : nan → 0

Group 2: 1 site(s)
----------------------------------------------------------------------

  Site: Gobabeb
    ✓ location_name       : N/A → Gobabeb
    ✓ country             : N/A → Namibia
    ✓ longitude           : N/A → 15.11956
    ✓ latitude            : N/A → -23.6002
 

In [29]:
# # This SITE_CONFIGS dictionary gives us our final list of sites and their order parameters for this request
# # SITE_CONFIGS = [
# #     {
# #         'sites': ['Albuquerque', 'Casablanca'],
# #         'order_parameters': {
# #             'ideal_num_acqs': 10,
# #             'request_type': 'timeseries',
# #             'assessment_domain': 'geometric'
# #         }
# #     },
# #     # Just examples of other configs
# #     {
# #         'sites': ['Baotou'],
# #         'order_parameters': {
# #             'ideal_num_acqs': 5,
# #             'request_type': 'other',
# #             'assessment_domain': 'geometric'
# #         }
# #     },
# #     {
# #         'sites': ['WLEF', 'PICS Libya-4'],
# #         'order_parameters': {
# #             'ideal_num_acqs': 3,
# #             'request_type': 'other',
# #             'assessment_domain': 'radiometric'
# #         }
# #     }
# # ]

# SITE_CONFIGS = [
#     {
#         "sites": ["PICS Libya-4"],
#         "order_parameters": {
#             "location_name": "PICS Libya-4",
#             "country": "Libya",
#             "longitude": 23.39,
#             "latitude": 28.55,
#             "remote_sensing_domain": "Optical Multi/Hyper",
#             "evaluation_category": "Radiometric",
#             "assessment_types": "Radiometric Calibration Quality (Absolute)",
#             "aoi_shape": "custom",
#             "max_aoi_cloud_pct": 0,
#             "max_scene_cloud_pct": 20,
#             "aoi_size_km": 3.0,
#             "max_view_angle": 10,
#             "min_num_acqs": 5,
#             "ideal_num_acqs": 5
#         }
#     },
#     {
#         "sites": ["Baotou"],
#         "order_parameters": {
#             "location_name": "China cal/val",
#             "country": "China",
#             "longitude": 109.629437,
#             "latitude": 40.851787,
#             "remote_sensing_domain": "Optical Multi/Hyper",
#             "evaluation_category": "Radiometric & Geometric",
#             "assessment_types": "Radiometric, Geometric Calibration Quality (LSF/MTF)",
#             "aoi_shape": "box",
#             "max_aoi_cloud_pct": 0,
#             "max_scene_cloud_pct": 0,
#             "aoi_size_km": 0.04,
#             "max_view_angle": 15,
#             "min_num_acqs": 1,
#             "ideal_num_acqs": 2
#         }
#     },
#     {
#         "sites": ["Rio Gallegos"],
#         "order_parameters": {
#             "location_name": "Argentina",
#             "country": "Argentina",
#             "longitude": -69.242713,
#             "latitude": -51.625811,
#             "remote_sensing_domain": "Optical Multi/Hyper",
#             "evaluation_category": "Geometric",
#             "assessment_types": "Geometric Calibration Quality (priority pointing site)",
#             "aoi_shape": "box",
#             "max_aoi_cloud_pct": 0,
#             "max_scene_cloud_pct": 15,
#             "aoi_size_km": 3.0,
#             "max_view_angle": 30,
#             "min_num_acqs": 5,
#             "ideal_num_acqs": 5
#         }
#     },
#     {
#         "sites": ["Albuquerque"],
#         "order_parameters": {
#             "location_name": "New Mexico",
#             "country": "USA",
#             "longitude": -106.613826,
#             "latitude": 35.068706,
#             "remote_sensing_domain": "Optical Multi/Hyper",
#             "evaluation_category": "Geometric",
#             "assessment_types": "Geometric Calibration Quality (priority pointing site, TS)",
#             "aoi_shape": "box",
#             "max_aoi_cloud_pct": 0,
#             "max_scene_cloud_pct": 15,
#             "aoi_size_km": 3.0,
#             "max_view_angle": 30,
#             "min_num_acqs": 10,
#             "ideal_num_acqs": 10
#         }
#     },
#     {
#         "sites": ["Catania"],
#         "order_parameters": {
#             "location_name": "Sicily",
#             "country": "Italy",
#             "longitude": 15.0654,
#             "latitude": 37.4699,
#             "remote_sensing_domain": "Optical Multi/Hyper",
#             "evaluation_category": "Geometric",
#             "assessment_types": "Geometric Calibration Quality (pointing site )",
#             "aoi_shape": "box",
#             "max_aoi_cloud_pct": 0,
#             "max_scene_cloud_pct": 15,
#             "aoi_size_km": 3.0,
#             "max_view_angle": 30,
#             "min_num_acqs": 5,
#             "ideal_num_acqs": 5
#         }
#     },
#     {
#         "sites": ["Casablanca"],
#         "order_parameters": {
#             "location_name": "Casablanca",
#             "country": "Morocco",
#             "longitude": -7.62242,
#             "latitude": 33.58037,
#             "remote_sensing_domain": "Optical Multi/Hyper",
#             "evaluation_category": "Geometric",
#             "assessment_types": "Geometric Calibration Quality (priority pointing site)",
#             "aoi_shape": "box",
#             "max_aoi_cloud_pct": 0,
#             "max_scene_cloud_pct": 15,
#             "aoi_size_km": 3.0,
#             "max_view_angle": 30,
#             "min_num_acqs": 5,
#             "ideal_num_acqs": 5
#         }
#     },
#     {
#         "sites": ["Singapore"],
#         "order_parameters": {
#             "location_name": "Singapore",
#             "country": "Singapore",
#             "longitude": 103.838974,
#             "latitude": 1.308939,
#             "remote_sensing_domain": "Optical Multi/Hyper",
#             "evaluation_category": "Geometric",
#             "assessment_types": "Geometric Calibration Quality (priority pointing site)",
#             "aoi_shape": "box",
#             "max_aoi_cloud_pct": 0,
#             "max_scene_cloud_pct": 15,
#             "aoi_size_km": 3.0,
#             "max_view_angle": 30,
#             "min_num_acqs": 5,
#             "ideal_num_acqs": 5
#         }
#     },
#     {
#         "sites": ["Valencia"],
#         "order_parameters": {
#             "location_name": "ESP-VLC2, El Palmar",
#             "country": "Spain",
#             "longitude": -0.3263,
#             "latitude": 39.2979,
#             "remote_sensing_domain": "Optical Multi/Hyper",
#             "evaluation_category": "Radiometric",
#             "assessment_types": "Radiometric Calibration Quality (Absolute)",
#             "aoi_shape": "circle",
#             "max_aoi_cloud_pct": 0,
#             "max_scene_cloud_pct": 20,
#             "aoi_size_km": 3.0,
#             "max_view_angle": 10,
#             "min_num_acqs": 1,
#             "ideal_num_acqs": 1
#         }
#     },
#     {
#         "sites": ["WLEF"],
#         "order_parameters": {
#             "location_name": "WLEF tower",
#             "country": "USA",
#             "longitude": -90.2732,
#             "latitude": 45.9449,
#             "remote_sensing_domain": "Optical Multi/Hyper",
#             "evaluation_category": "Radiometric",
#             "assessment_types": "Radiometric Calibration Quality (Absolute)",
#             "aoi_shape": "circle",
#             "max_aoi_cloud_pct": 0,
#             "max_scene_cloud_pct": 20,
#             "aoi_size_km": 3.0,
#             "max_view_angle": 10,
#             "min_num_acqs": 1,
#             "ideal_num_acqs": 1
#         }
#     },
#     {
#         "sites": ["Gobabeb"],
#         "order_parameters": {
#             "location_name": "Gobabeb",
#             "country": "Namibia",
#             "longitude": 15.11956,
#             "latitude": -23.6002,
#             "remote_sensing_domain": "Optical Multi/Hyper",
#             "evaluation_category": "Radiometric",
#             "assessment_types": "Radiometric Calibration Quality (Absolute)",
#             "aoi_shape": "box",
#             "max_aoi_cloud_pct": 0,
#             "max_scene_cloud_pct": 20,
#             "aoi_size_km": 3.0,
#             "max_view_angle": 10,
#             "min_num_acqs": 5,
#             "ideal_num_acqs": 10
#         }
#     },
#     {
#         "sites": ["Railroad Valley"],
#         "order_parameters": {
#             "location_name": "Nevada",
#             "country": "USA",
#             "longitude": -115.69,
#             "latitude": 38.497,
#             "remote_sensing_domain": "Optical Multi/Hyper",
#             "evaluation_category": "Radiometric",
#             "assessment_types": "Radiometric Calibration Quality (Absolute)",
#             "aoi_shape": "box",
#             "max_aoi_cloud_pct": 0,
#             "max_scene_cloud_pct": 20,
#             "aoi_size_km": 3.0,
#             "max_view_angle": 10,
#             "min_num_acqs": 5,
#             "ideal_num_acqs": 10
#         }
#     },
#     {
#         "sites": ["Golmud"],
#         "order_parameters": {
#             "location_name": "Golmud",
#             "country": "China",
#             "longitude": 94.3286,
#             "latitude": 36.3977,
#             "remote_sensing_domain": "Optical Multi/Hyper",
#             "evaluation_category": "Radiometric",
#             "assessment_types": "Radiometric Calibration Quality (Absolute)",
#             "aoi_shape": "box",
#             "max_aoi_cloud_pct": 0,
#             "max_scene_cloud_pct": 20,
#             "aoi_size_km": 3.0,
#             "max_view_angle": 10,
#             "min_num_acqs": 1,
#             "ideal_num_acqs": 5
#         }
#     }
# ]


### Create and write subset GeoJSON for this request

In [30]:
sites_subset = sites_updated[sites_updated['Site Name'].isin(SITES_FOR_REQUEST)]

In [31]:
DROP_COLS = ['Site Name abbrev',
             'Location Name',
 'Country',
 'Program Use',
 'Longitude',
 'Latitude',
 'Remote Sensing Domain',
 'Priority Level',
 'Evaluation Category',
 'Source',
 'Surface Domain',
 'Assessment type(s)',
 'Resolution Category',
 'aoi_shape',
 'aoi_size_km',
 # 'max_view_angle',
 # 'min_num_acqs',
 # 'ideal_num_acqs',
 'max_cloud_pct_aoi',
 'max_cloud_pct_scene',
 'contact',
 'reference',
 'notes',
 'area_km2',
 'feature_type',
 'parent_site',
 ' Location',
 'cr:id',
 'cr:lat',
 'cr:lon',
 'cr:height_above_ellipsoid_m',
 'cr:orientation_deg',
 'cr:elevation_angle_d',
 'cr:size_m',
 'geometry',
 'location_name',
 'country',
 'longitude',
 'latitude'
            ]

In [32]:
#sites_subset.drop(DROP_COLS, axis=1, inplace=True)

In [33]:
sites_subset.columns.to_list()

['Site Name',
 'geometry',
 'location_name',
 'country',
 'longitude',
 'latitude',
 'remote_sensing_domain',
 'evaluation_category',
 'assessment_types',
 'aoi_shape',
 'max_aoi_cloud_pct',
 'max_scene_cloud_pct',
 'aoi_size_km',
 'max_view_angle',
 'min_num_acqs',
 'ideal_num_acqs']

In [34]:
#sites_subset.head()

,Site Name,geometry,location_name,country,longitude,latitude,remote_sensing_domain,evaluation_category,assessment_types,aoi_shape,max_aoi_cloud_pct,max_scene_cloud_pct,aoi_size_km,max_view_angle,min_num_acqs,ideal_num_acqs
10,Baotou,"POLYGON ((109.62968 40.85161, 109.62967 40.851...",China cal/val,China,109.629437,40.851787,Optical Multi/Hyper,Radiometric & Geometric,,box,0.0,0.0,0.04,15.0,1.0,0.0
31,Gobabeb,"POLYGON ((15.14897 -23.60017, 15.14883 -23.602...",Gobabeb,Namibia,15.119560,-23.600200,Optical Multi/Hyper,Radiometric,,circle,0.0,0.0,3.00,10.0,5.0,10.0
48,La Crau,"POLYGON ((4.90129 43.55828, 4.90103 43.55563, ...",La Crau,France,4.864167,43.558889,Optical Multi/Hyper,Radiometric & Geometric,,circle,0.0,0.0,3.00,10.0,0.0,0.0
51,PICS Libya-4,"POLYGON ((23.39665 28.55833, 23.39110 28.53333...",PICS Libya-4,Libya,23.390000,28.550000,Optical Multi/Hyper,Radiometric,,custom,0.0,0.0,3.00,0.0,5.0,10.0
53,Railroad Valley,"POLYGON ((-115.65561 38.49661, -115.65582 38.4...",Nevada,USA,-115.690000,38.497000,Optical Multi/Hyper,Radiometric,,circle,0.0,0.0,3.00,10.0,5.0,10.0


In [38]:
#sites_subset.explore()

In [36]:
OUTPUT_DIR = '/home/pmontesa/code/csda_summaries/sites' # Specify your output dir here

In [37]:
sites_subset.to_file(f'{OUTPUT_DIR}/csda_sites_aoi_{VENDOR_NAME}_{DATE}.geojson')